# EX6 - Intermediate Representations and Analysis with `Lark`

## Grammar Engineering

## MEI/2026-27

#### Nuno Macedo
Universidade do Minho

The following exercises are intended to be solved with pen and paper and implemented with a grammar processor for Python, [Lark](https://lark-parser.readthedocs.io).

# A simple imperative language

- Consider a simple language with
  - integer variable assignments
  - `if` branches (with optional `else`)
  - `while` loops
  - `print` and `input` statements

- Code blocks have mandatory brackets `{` `}`

- This language can be defined by the `Lark` grammar below

- *Note*: it is disregarding operator precedence and typing rules

In [ ]:
grammar = """
?start: block

?block: stmt+

?stmt: assign_stmt ";"
     | if_stmt
     | while_stmt
     | print_stmt ";"

assign_stmt: CNAME "=" expr

if_stmt: "if" "(" expr ")" "{" block "}" ("else" "{" block "}")?

while_stmt: "while" "(" expr ")" "{" block "}"

print_stmt: "print" expr

?expr: bool
     | "input"
     | CNAME -> var
     | NUMBER -> number
     | expr "+" expr -> add
     | expr "-" expr -> sub
     | expr "*" expr -> mul
     | expr "<" expr -> lt
     | expr ">" expr -> gt
     | "(" expr ")"

bool : "true" | "false"

%import common.CNAME
%import common.NUMBER
%import common.WS
%import common.BOOL
%ignore WS
"""

# Exercise 1

Consider the simple imperative language defined above, and draw the *control-flow graph* (CFG) for the following code snippets.

## Exercise 1.1

In [ ]:
ex_1_1 = """
x = 1;
if (x < 10) {
    y = x + 1;
} else {
    y = 0;
}
print y;
"""

## Exercise 1.2



In [ ]:
ex_1_2 = """
x = input;
y = x + 5;
if (y > 12) {
    z = x + y;
    z = 2 * z;
}
else {
    z = y - 2;
    print z;
}
while (z > 10) {
    z = z - 1;
    x = x + 2;
}
print z;
"""

## Exercise 1.3


In [ ]:
ex_1_3 = """
x = input;
y = x + 5;
if (y > 12) {
    if (x < 20) {
        z = 2 * z;
    }
}
else {
    if (x < 20) {
        print x;
    } else {
      print x * 2;
    }
}
print 0;
"""

## Exercise 1.4


In [ ]:
ex_1_4 = """
x = input;
while (x > 0) {
  if (x > 5) {
    print 5;
  } else {
    print 10;
  }
}
print 0;
"""

# Exercise 2

Consider the code provided below, that uses a `Lark` `Transformer` to extract a CFG from the parsing trees of programs written in the given grammar.
  
It relies on a standard Python library for graphs to create, manipulate and draw the CFG, [`NetworkX`](https://networkx.org/).


Extend the language above, the corresponding grammar, and the CFG extraction `Transformer`, to support additional control flow constructs.

In [ ]:
!apt-get -qq install graphviz > /dev/null
!pip install lark networkx graphviz pydot

In [ ]:
import networkx as nx
from networkx.drawing.nx_pydot import write_dot

class CFGFragment:
    """A CFG fragment with an entry and an exit."""

    def __init__(self, entry, exit, graph=None):
        self.entry = entry
        self.exit = exit
        self.graph = graph if graph is not None else nx.DiGraph()

    def merge(self, fragment):
        """Connect two fragments, merging consecutive basic blocks."""

        self.graph.add_nodes_from(fragment.graph.nodes(data=True))
        self.graph.add_edges_from(fragment.graph.edges(data=True))

        current = self.exit
        next_block = fragment.entry

        if (self.graph.nodes[current]["type"] == "basic"
                and self.graph.nodes[next_block]["type"] == "basic"):

            # Append instructions
            self.graph.nodes[current]["stmt"] += (
                "\n" + self.graph.nodes[next_block]["stmt"]
            )

            self.graph.nodes[current]["label"] = (
                f"{current}\n{self.graph.nodes[current]['stmt']}"
            )

            # Redirect incoming edges
            for pred in list(self.graph.predecessors(next_block)):
                if pred != current:
                    self.graph.add_edge(pred, current)

            # Redirect outgoing edges
            for succ in list(self.graph.successors(next_block)):
                if succ != current:
                    self.graph.add_edge(current, succ)

            self.graph.remove_node(next_block)

            if fragment.exit != next_block:
                self.exit = fragment.exit

        else:
            self.graph.add_edge(current, next_block)
            self.exit = fragment.exit

In [ ]:
from lark import Lark, Transformer

class CFGTransformer(Transformer):

    def __init__(self):
        self.counter = 0

    def new_block(self, stmt="", block_type="basic"):
        name = f"B{self.counter}"
        self.counter += 1

        graph = nx.DiGraph()
        graph.add_node(
            name,
            stmt=stmt,
            type=block_type,
            label=f"{name}\n{stmt}" if stmt else name
        )

        return CFGFragment(name, name, graph)

    def block(self, items):
        fragment = items[0]

        for item in items[1:]:
            fragment.merge(item)

        return fragment

    def assign_stmt(self, items):
        return self.new_block(f"{items[0].value} = ...")

    def print_stmt(self, items):
        return self.new_block("print ...")

    def if_stmt(self, items):
        condition = self.new_block("if ...", "condition")
        then_branch = items[1]
        merge = self.new_block()

        condition.graph.add_nodes_from(then_branch.graph.nodes(data=True))
        condition.graph.add_edges_from(then_branch.graph.edges(data=True))

        condition.graph.add_edge(condition.entry, then_branch.entry)
        condition.graph.add_edge(then_branch.exit, merge.entry)

        if len(items) > 2:
            else_branch = items[2]

            condition.graph.add_nodes_from(else_branch.graph.nodes(data=True))
            condition.graph.add_edges_from(else_branch.graph.edges(data=True))

            condition.graph.add_edge(condition.entry, else_branch.entry)
            condition.graph.add_edge(else_branch.exit, merge.entry)

        else:
            condition.graph.add_edge(condition.entry, merge.entry)

        condition.graph.add_nodes_from(merge.graph.nodes(data=True))

        return CFGFragment(
            condition.entry, merge.exit, condition.graph
        )

    def while_stmt(self, items):
        condition = self.new_block("while ...", "condition")
        body = items[1]
        loop_exit = self.new_block()

        condition.graph.add_nodes_from(body.graph.nodes(data=True))
        condition.graph.add_edges_from(body.graph.edges(data=True))
        condition.graph.add_nodes_from(loop_exit.graph.nodes(data=True))

        condition.graph.add_edge(condition.entry, body.entry)
        condition.graph.add_edge(body.exit, condition.entry)
        condition.graph.add_edge(condition.entry, loop_exit.entry)

        return CFGFragment(
            condition.entry, loop_exit.exit, condition.graph
        )

In [ ]:
from networkx.drawing.nx_pydot import to_pydot
from graphviz import Source

parser = Lark(grammar)

def extract_cfg(source):
    tree = parser.parse(source)
    return CFGTransformer().transform(tree)


def draw_cfg(source):
    fragment = extract_cfg(source)
    return Source(to_pydot(fragment.graph).to_string())

## Exercise 2.1

Introduce in the language `for` loops of the shape.


In [ ]:
ex_2_1 = """
i = 0;
for (i = 10; i < 10; i = i + 1) {
    print i;
}
print i;
"""

## Exercise 2.2


Introduce in the language additional control flow constructs statements, namely `break` and `continue` statements for loops.


In [ ]:
ex_2_2 = """
while (z < 10) {
    if (z == 0) {
        break;
    } else {
        if (z > 10) {
            continue;
        }
    }
    z = z - 1;
    print z;
}
"""

## Exercise 2.3

Introduce in the language `return` statements that immediately terminate a procedure.

In [ ]:
ex_2_3 = """
x = 10;
if (x > 20) {
    return x;
}
while (x > 10) {
    x = x / 2;
    if (x == 0) {
        return 0;
    }
}
"""

# Exercise 3

Consider the provided code fragments below, calculate their CFGs, and identify unreachable code.


## Exercise 3.1

In [ ]:
code_ex3_1 = """
x = 10;
y = x + 5;
if (y > 12) {
    return z;
    x = x * 2;
}
print 0;
"""

## Exercise 3.2

In [ ]:
code_ex3_2 = """
x = 10;
y = x + 5;
if (y > 12) {
    if (x < 20) {
        return z;
        if (x < 10) {
            z = 2 * z;
        }
    }
}
else {
    if (x < 20) {
        print x;
    } else {
        print x * 2;
    }
}
return x;
print 0;
"""

## Exercise 3.3


In [ ]:
code_ex3_3 = """
x = 10;
i = 0;
while (i < 100) {
    if (i == x) {
        print x;
        break;
    } else {
        print x;
        continue;
    }
    print x - i;
}
print 0;
"""

# Exercise 4

Use the CFG extracted from source code to perform basic static analyses.

*Hint*: explore the graph traversals provided by `NetworkX`.



## Exercise 4.1

Use the CFG to detect *unreachable code*, i.e., blocks that are not reachable in the CFG. Assume the language extension that contains at least `return` statements.

*Hint*: use `nx.descendants()`.


## Exercise 4.2

Use the CFG to identify *cycles*, i.e., paths that lead back to a previously visited node.

*Hint*: explore the cycle-detection algorithms provided by `NetworkX`.

In [ ]:
code_ex4_2 = """
x = 10;
y = x + 5;
while (y > 12) {
    if (x < 20) {
        if (x < 10) {
            z = 2 * z;
        }
    }
}
return x;
print 0;
"""

# Exercise 5

Consider the simple imperative language defined above, and draw the *data-flow graph* (DFG) for the following code snippets.

## Exercise 5.1


In [ ]:
ex_5_1 = """
i = 0;
sum = 0;
while (i < 3) {
    sum = sum + i;
    i = i + 1;
}
print sum;
"""

## Exercise 5.2


In [ ]:
ex_5_2 = """
x = 0;
i = 0;
while (i < 5) {
    if (i % 2 == 0) {
        x = x + 1;
    } else {
        x = x - 1;
    }
    i = i + 1;
}
print x;
"""

## Exercise 5.3


In [ ]:
ex_5_3 = """
x = 1;
y = 2;

if (cond) {
    x = x + y;
}

z = x * 2;
print z;
"""

## Exercise 5.4


In [ ]:
ex_5_4 = """
a = 1;
b = 2;
if (cond) {
    a = b + 3;
    c = a + 1;
} else {
    a = a + 2;
    c = a + 4;
}
d = c + b;
return d;
"""

# Exercise 6

Consider the code fragments below. For each fragment:

  - Construct the corresponding DFG identifying the data dependencies between statements
  - Identify the dead code (assignments whose results do not contribute to the program's observable behaviour)
  - Rewrite the fragment after eliminating the dead code

## Exercise 6.1


In [ ]:
ex_6_1 = """
a = 1;
b = a + 2;
c = b * 3;
d = 5;
print a;
"""

## Exercise 6.2


In [ ]:
ex_6_2 = """
i = 0;
sum = 0;
while (i < 5) {
    x = i * 2;
    y = x + 1;
    z = y + 3;
    sum = sum + i;
    i = i + 1;
}
print sum;
"""

## Exercise 6.3



In [ ]:
ex_6_3 = """
x = 0;
y = 0;
z = 0;
i = 0;
while (i < 3) {
    x = x + i;
    if (i % 2 == 0) {
        y = y + x;
        t = y * 2;
    }
    i = i + 1;
}
print x + y;
"""

# Exercise 7

Consider the code fragments below. For each fragment:

- Construct the corresponding DFG
- Perform constant propagation, identifying variables and expressions whose values are known constants
- Based on the analysis, identify possible code simplifications

## Exercise 7.1


In [ ]:
ex_7_1 = """
y = 0;

if (flag > 0) {
    y = 8 - 3;
} else {
    y = 2 + 3;
}

z = y * 2;
print z;
"""

## Exercise 7.2


In [ ]:
ex_7_2 = """
x = 4;
y = 2;

if (flag > 0) {
    x = 5;
}

z = x + y;
print z;
"""

## Exercise 7.3


In [ ]:
ex_7_3 = """
step = 2;
limit = 3;
i = 0;
x = 0;

while (i < limit) {
    x = x + step;
    i = i + 1;
}

z = x + step;
print z;
"""

# Exercise 8

Consider the code fragments below. For each fragment:

- Construct the corresponding DFG, identifying the data dependencies between statements
- Extend the representation with control dependencies to obtain a *program dependence graph* (PDG)
- Perform a backward slice with respect to the slicing criterion `print w`, identifying all statements that may influence its result or execution


## Exercise 8.1


In [ ]:
ex_8_1 = """
x = 1;
y = 2;
z = 0;
if (x < y) {
    x = x + y;
    z = x + 1;
}
w = z + 2;
print w;
"""

## Exercise 8.2



In [ ]:
ex_8_2 = """
x = 0;
y = 1;
z = 0;

i = 0;
while (i < 4) {
    if (i % 2 == 0) {
        x = x + i;
    } else {
        y = y + i;
    }
    z = x + y;
    i = i + 1;
}

w = z * 2;
print w;
"""

# Exercise 9 (optional)

Inspired by the provided `Lark` `Transformer` to extract a CFG, write a `Transformer` to extract the DFG for programs written in the simple imperitive language.

Start by identifying all definitions and uses of a block, and then use the CFG to determine which definitions reach which uses.

*Hint*: explore the graph traversals provided by `NetworkX`.



-- nmm, 2026-11-19